# Assignment 3 - Big Data Analytics using Apache Spark
## Student: Turgunboev Dadakhon
Description: Web server logs capture data about each registered event. They contain valuable information about site visitors, their behavior, the actions of search robots, business intelligence, and potential security threats.
* Dataset: Nginx server access log(3.26 GB) for 2019 year.
    * Columns: 'remote_addr', 'time_local', 'request', 'url', 'status', 'body_bytes_sent'
    * Sum of requests: 10269414   
    
* Assignment Tasks:
    * Exercises on Spark RDD
    * Exercises on Spark DataFrame
    * Exercises on Spark SQL
    * Exercises on Spark MLlib
* Prerequisites
    * Access to the university cluster

## Task 1: Exercises on Spark RDD
 1. Parses a single line of Nginx access log into structured fields.
 2. Create RDD with GET requests count per year
 3. Create an RDD that contains the unique ip addresses that has top 10 most successful GET requests.

In [ ]:
# Import the SparkSession class from PySpark SQL module
# This is the main entry point for DataFrame and SQL functionality
from pyspark.sql import SparkSession

# Create a SparkSession instance or get an existing one
# SparkSession is the unified entry point for Spark functionality
spark = SparkSession.builder.appName("Dadakhon-Assignment3").getOrCreate()

# SparkContext is the main connection to Spark cluster and represents the entry point for low-level APIs
sc = spark.sparkContext

# Set the log level to ERROR to reduce console output
# This suppresses INFO and WARN messages, showing only ERROR messages
sc.setLogLevel('ERROR')

# Print confirmation message
print("Spark session created successfully!")

In [ ]:
# Create an RDD (Resilient Distributed Dataset) by reading the log file
log_rdd = sc.textFile("data/dadakhon/access.log")
log_rdd.take(4)

### 1. Parses a single line of Nginx access log into structured fields.
* Args:
        line (str): A raw log line from Nginx access log
* Returns:
        dict: Parsed log fields as a dictionary, or None if parsing fails

In [ ]:
import re
from datetime import datetime

# Parse a line of nginx access log
def parse_log_line(line):
    pattern = (
        r'(?P<remote_addr>\S+) - - '                # Client IP address
        r'\[(?P<time_local>.*?)\] '                 # Timestamp in brackets
        r'"(?P<request_method>\S+) '                # HTTP method (GET/POST/etc)
        r'(?P<url>\S+) \S+" '                       # Request URL
        r'(?P<status>\d{3}) '                       # HTTP status code
        r'(?P<body_bytes_sent>\d+)'                # Response body size in bytes
    )

    # Attempt to match the pattern against the log line    
    match = re.search(pattern, line)
    if match:
        # Extract and parse the timestamp (ignoring timezone)
        time_str = match.group("time_local").split()[0]  
        dt = datetime.strptime(time_str, "%d/%b/%Y:%H:%M:%S")
        
        # Regular expression pattern to match Nginx log components
        return {
            "remote_addr": match.group("remote_addr"),
            "time_local": {
                "year": dt.year,
                "month": dt.month,
                "day": dt.day,
                "hours": dt.hour,
                "minutes": dt.minute,
                "seconds": dt.second
            },
            "request": match.group("request_method"),
            "url": match.group("url"),
            "status": int(match.group("status")),
            "body_bytes_sent": int(match.group("body_bytes_sent"))
        }
    else:
        # Return None for lines that don't match the pattern
        return None

In [ ]:
# Apply parser and filter out None values
parsed_rdd = log_rdd.map(parse_log_line).filter(lambda x: x is not None)

# Get the count of items in the parsed RDD
total_records = parsed_rdd.count()
print(f"Total number of log entries: {total_records}")

# Collect and display first 10 parsed records
parsed_rdd.take(10)

### 2. Create RDD with GET requests count per year
1. Show the first 10 results from the RDD sorted by most recent years. 

In [ ]:
# Create RDD with GET requests count per year
get_requests_by_year = (
    parsed_rdd
    .filter(lambda x: x['request'] == 'GET')  # Filter only GET requests
    .map(lambda x: (x['time_local']['year'], 1))  # Create (year, 1) pairs
    .reduceByKey(lambda a, b: a + b)  # Count requests per year
    .sortBy(lambda x: -x[0])  # Sort by year (descending)
)

# Proof of implementation - show first 10 results
print("GET requests per year (most recent first):")
for year, count in get_requests_by_year.take(10):
    print(f"{year}: {count} requests")
    
unique_years = parsed_rdd.map(lambda x: x['time_local']['year']).distinct().collect()

In [ ]:
unique_years

The analyzed dataset contains Nginx web server access logs with entries exclusively from 2019. This was confirmed through comprehensive examination of all timestamp records in the dataset.

### 3. Create an RDD that contains the unique ip addresses that has top 10 most successful GET requests.
1. Show the first 10 results from the RDD sorted by top 10 most successful GET requests…

In [ ]:
# Create RDD with top successful IPs
top_successful_ips = (
    parsed_rdd
    .filter(lambda x: x['request'] == 'GET' and x['status'] == 200)  # Successful GETs
    .map(lambda x: (x['remote_addr'], 1))  # Create (IP, 1) pairs
    .reduceByKey(lambda a, b: a + b)  # Count successful requests per IP
    .sortBy(lambda x: -x[1])  # Sort by count (descending)
)

# Proof of implementation - show top 10 results
print("\nTop 10 IPs with most successful GET requests:")
for ip, count in top_successful_ips.take(10):
    print(f"{ip}: {count} successful requests")

Result: 

In this task, we processed the parsed RDD to identify the IP addresses with the most successful GET requests (HTTP status 200).
We filtered the data to include only successful GET operations, mapped each record to a pair (IP, 1), and aggregated the counts using reduceByKey.
Finally, we sorted the results in descending order and displayed the top 10 IPs.
The output reveals that a few IP addresses, such as 66.249.66.194 and 66.249.66.91, are responsible for a very high number of successful requests.
This pattern suggests automated access behavior, possibly from web crawlers (e.g., Googlebot) or extremely active users.

### Summary task 1:
In this task, we successfully set up a PySpark environment using SparkSession and SparkContext. We then loaded a large Nginx access log file into an RDD and implemented a custom parser using regular expressions to extract key fields from each log entry. The parsing function handled timestamps, HTTP methods, URLs, status codes, and response sizes, while skipping invalid lines.

After parsing, we:
 * Counted the total number of valid log entries.
 * Analyzed the number of GET requests per year.
 * Identified the top 10 IP addresses that made the most successful GET (HTTP 200) requests.

The data revealed that all requests in the dataset were from the year 2019, with approximately 10 million GET requests logged. The IP address 66.249.66.194 made the highest number of successful GET requests, with over 300,000 successful hits.

Overall, this step provided a clean, structured representation of the raw logs and extracted important high-level statistics useful for further analysis.

## Task 2: Exercises on Spark DataFrame

1. Createa Spark DataFrame with appropriate column names and data types based on the parsed data.
2. Filter the DataFrame to show only the log entries with a status code of 200.
3. Find all requests that resulted in a 4xx error (status code between 400 and 499).
4. Categorize the requests based on the status code (e.g., “Success” for 2xx, “Client Error” for 4xx, “Server Error” for 5xx).
5. Calculate the total body_bytes_sent by each remote_addr.
6. Analyse the linear correlation between total number of successful requests and the average body size for the logs per year.
7. Define a UDF for extracting the first part from the request endpoint.

### 1. Createa Spark DataFrame with appropriate column names and data types based on the parsed data.
   * Show the size od the dataframe (rows and columns).
   * Show the schema of the dataframes after reading.
   * Show first 10 samples of the dataframes.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, StructType

# Define the schema for the parsed log data
# The "time_local" field is a nested struct containing year, month, day, etc.
schema = StructType([
    StructField("remote_addr", StringType(), True),
    StructField("time_local", StructType([
        StructField("year", IntegerType(), True),
        StructField("month", IntegerType(), True),
        StructField("day", IntegerType(), True),
        StructField("hours", IntegerType(), True),
        StructField("minutes", IntegerType(), True),
        StructField("seconds", IntegerType(), True)
    ]), True),
    StructField("request", StringType(), True),
    StructField("url", StringType(), True),
    StructField("status", IntegerType(), True),
    StructField("body_bytes_sent", IntegerType(), True)
])

# Create a DataFrame from the parsed RDD using the defined schema
log_df = spark.createDataFrame(parsed_rdd, schema)

# 1. Print the size of the DataFrame: number of rows and columns
print(f"Size of Dataframes: {log_df.count()} rows, {len(log_df.columns)} columns")

In [ ]:
# 2. Display the structure (schema) of the DataFrame
log_df.printSchema()

In [ ]:
# 3. Show the first 10 rows of the DataFrame without truncating column values
log_df.show(10, truncate=False)

Subtask Summary: 

In this subtask of Task 2, we converted the parsed RDD into a structured PySpark DataFrame using a custom schema.
The time_local field was defined as a nested struct with year, month, day, hour, minute, and second components.
We then validated the DataFrame by checking its size (10,269,414 rows and 6 columns), printing its schema, and displaying sample rows.
This step sets up the structured data format needed for further analysis in Task 2.